# Distillation + Training (crash-safe) · Skincare Advisor

**这一版针对一件事重新设计:Colab 运行时被回收后不重复花钱。**

前一版把备份放在最后,结果 D1 跑完、运行时回收,676 条蒸馏数据(约 $0.20)全没了。
这一版做了三处改动:

| 改动 | 作用 |
|---|---|
| **第 1 格就挂 Drive 并定义 `backup()`** | 备份能力在任何耗时操作之前就绪 |
| **每一步自动从 Drive 恢复** | 断线重跑时,已完成的部分直接复用,不重算不重付 |
| **蒸馏缓存优先恢复** | `sft.cache.jsonl` 在 Drive 里的话,已调用过的 API 一次都不会重复调 |

另外改用**真实产品库**(2,282 个 Sephora 产品)而不是 12 个合成产品。

> **断线了怎么办:重连后从第 1 格重新全部运行。** 已完成的步骤会自动跳过或从 Drive 恢复。


## 1. Drive + 备份能力(**必须第一个跑**)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

WORK       = Path('/content/skincare')
SRC        = WORK / 'data' / 'processed'
MODEL_SRC  = WORK / 'models' / 'llm'
DATA_DST   = Path('/content/drive/MyDrive/skincare_data');   DATA_DST.mkdir(parents=True, exist_ok=True)
MODEL_DST  = Path('/content/drive/MyDrive/skincare_models'); MODEL_DST.mkdir(parents=True, exist_ok=True)
INDEX_DST  = DATA_DST / 'index'


def backup():
    """把当前产物同步到 Drive。每完成一步就调用一次。"""
    n = 0
    for pat in ('*.jsonl', '*.json'):
        for f in SRC.glob(pat):
            shutil.copy(f, DATA_DST / f.name); print(f'  data   {f.name}'); n += 1
    idx = SRC / 'index'
    if idx.exists():
        shutil.copytree(idx, INDEX_DST, dirs_exist_ok=True); print('  index  (FAISS)'); n += 1
    for name in ('sft-lora', 'grpo'):
        d = MODEL_SRC / name
        if d.exists():
            shutil.copytree(d, MODEL_DST / name, dirs_exist_ok=True); print(f'  model  {name}'); n += 1
    print(f'backup done -> Drive ({n} items)')


def restore():
    """从 Drive 恢复已有产物。断线重跑时这一步让你不必重算。"""
    SRC.mkdir(parents=True, exist_ok=True)
    n = 0
    for f in list(DATA_DST.glob('*.jsonl')) + list(DATA_DST.glob('*.json')) + list(DATA_DST.glob('*.parquet')):
        shutil.copy(f, SRC / f.name); print(f'  restored {f.name}'); n += 1
    if INDEX_DST.exists():
        shutil.copytree(INDEX_DST, SRC / 'index', dirs_exist_ok=True); print('  restored index'); n += 1
    for name in ('sft-lora', 'grpo'):
        d = MODEL_DST / name
        if d.exists():
            MODEL_SRC.mkdir(parents=True, exist_ok=True)
            shutil.copytree(d, MODEL_SRC / name, dirs_exist_ok=True); print(f'  restored {name}'); n += 1
    print(f'restore done ({n} items)')


print('Drive 已挂载,backup() / restore() 就绪')
print('\nDrive 现有内容:')
for f in sorted(DATA_DST.iterdir()): print('  data  ', f.name)
for f in sorted(MODEL_DST.iterdir()): print('  model ', f.name)


## 2. 代码与依赖

**第一次跑之前**,请先把这两个文件上传到 Drive 的 `MyDrive/skincare_data/`:
`products.parquet` 和 `chunks.parquet`(在你本机 `~/Documents/skincare/data/processed/`)。


In [ ]:
import os, subprocess

if not (WORK / 'pyproject.toml').exists():
    subprocess.run('git clone -q https://github.com/Shenghan-Gao/skincare-advisor.git /content/skincare',
                   shell=True, check=True)
else:
    subprocess.run(f'cd {WORK} && git pull -q', shell=True)
os.chdir(WORK)

os.system('pip install -q -e ".[dev,rag,llm]" 2>&1 | tail -2')
import importlib.util
if importlib.util.find_spec('torchao') is not None:
    print('卸载旧版 torchao(与新版 PEFT 冲突)')
    os.system('pip uninstall -y -q torchao')

restore()          # 把 Drive 上已有的东西拉回来

import torch
print(f"\nGPU {torch.cuda.is_available()} | "
      f"{'bf16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'fp16'}")
assert torch.cuda.is_available(), '没有 GPU:代码执行程序 -> 更改运行时类型 -> T4 GPU'
assert (SRC / 'products.parquet').exists(), \
    'Drive 的 skincare_data/ 里缺 products.parquet —— 请先从本机上传'
assert (SRC / 'chunks.parquet').exists(), \
    'Drive 的 skincare_data/ 里缺 chunks.parquet —— 请先从本机上传'
print('✅ 代码、依赖、真实产品库就绪')


## 3. FAISS 索引(约 20–30 分钟;Drive 里有就自动跳过)


In [ ]:
if (SRC / 'index' / 'chunks.faiss').exists():
    print('索引已存在(从 Drive 恢复),跳过重建')
else:
    !python -m skincare.rag.index
    backup()

!python scripts/verify_handoff.py rag


## 4. 教师蒸馏(真实产品库)

注意这里**没有 `--mock-retrieval`** —— 走的是刚建好的真实索引。
`sft.cache.jsonl` 若已从 Drive 恢复,已调用过的 API 会命中缓存,**不重复付费**。


In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

N_FULL = 800
!python -m skincare.llm.data_build --n $N_FULL --mode both --inspect 2 2>&1 | tail -40
backup()


## 5. LoRA SFT


In [ ]:
BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
os.environ['BASE'] = BASE
SFT_OUT = '/content/skincare/models/llm/sft-lora'

!python -m skincare.llm.sft_lora --base $BASE --epochs 2 --bs 1 --accum 8 --max-len 2048 --out $SFT_OUT 2>&1 | tail -12

assert any(Path(SFT_OUT).glob('adapter*')), 'SFT 没产出 adapter'
backup()


## 6. GRPO

重点看输出里 `rewards/xxx_reward/mean` 五行**是否随步数上升** —— 那是报告的头号结果。


In [ ]:
GRPO_OUT = '/content/skincare/models/llm/grpo'
!python -m skincare.llm.grpo_train --base $BASE --adapter $SFT_OUT \
    --steps 300 --group-size 8 --accum 4 --max-completion-length 512 --out $GRPO_OUT 2>&1 | tail -22

assert any(Path(GRPO_OUT).glob('adapter*')), 'GRPO 没产出 adapter'
backup()


## 7. 更新 manifest(交给组员 C 做评估的唯一接口)


In [ ]:
import json
mf = WORK / 'models' / 'llm' / 'manifest.json'
m = json.loads(mf.read_text())
m['base'] = BASE
m['sft']  = str(MODEL_DST / 'sft-lora')
m['grpo'] = str(MODEL_DST / 'grpo')
mf.write_text(json.dumps(m, ensure_ascii=False, indent=1))
shutil.copy(mf, MODEL_DST / 'manifest.json')
print(mf.read_text())

print('\n' + '='*56)
print('  完成。交给组员 C 的东西都在 Drive/skincare_models/:')
print('    sft-lora/   grpo/   manifest.json')
print('\n  她跑这条命令即可产出报告要的三段式对比表:')
print('    python -m skincare.eval.run_eval --split data/processed/rl_test.jsonl \\')
print('           --variants base sft grpo')
